# Atom-based: total W + GHZ key rate

Level 0 returns W only. Levels 1 and 2 return **W + GHZ**, with one common clock and one shared brightness setting.

Keep the supplied `helper.py` next to this notebook and set `SOURCE_DIR` to your existing `.dill` files. Merging, corrections, and robust searches are imported. The source expressions are unchanged.

All times are in microseconds; rates are in bits/s. The calculation retains the previous $11/6$ and factor-3 time bounds, and sums the accepted routes as
$$R_{\rm total}=10^6\frac{\sum_r p_r f_r}{C_{\rm cycle}+P_{\rm accepted}t_{\rm meas}}.$$


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import helper as hp

SOURCE_DIR = Path(".")
POSTPROCESSING = "best"   # "one_way", "ad" (forced), or "best" (optional)
FINAL_POOLING = "route"      # "route" or "class"
C = 0.2                     # km / microsecond
bounds = [(0.0, 1.0)]  # q
SEARCH_OPTIONS = dict(n_log=201, n_linear=101, log_min=1e-8,
                      grid_refinements=1, warn_no_positive=False)


## Source expressions

In [2]:
argument_names = ("q", "p_r", "p_l")
get_prob_click = hp.load_source_function(
    SOURCE_DIR / "atom_based_prob_click.dill", argument_names)
get_ion_dm = hp.load_source_function(
    SOURCE_DIR / "atom_based_ion_dm.dill", argument_names)


## Overall key rate

In [3]:
def get_key_rate_components(vars, d, p_l_val, num_level=2, *,
                            postprocessing=None, pooling=None, return_states=False):
    """Total, W, and GHZ contributions at one common source setting."""
    mode = POSTPROCESSING if postprocessing is None else postprocessing
    pool = FINAL_POOLING if pooling is None else pooling
    q_val, = map(float, vars)
    d_ES = d / (2 ** (num_level + 1))
    t_signal = 2.0 * d_ES / C
    p_r_val = hp.get_loss_ratio(d_ES)
    args = (q_val, p_r_val, p_l_val)
    p_EL = hp.probability(get_prob_click(*args), "click")
    attempt_time = 100.0 + t_signal
    elementary = dict(prob_click=p_EL, attempt_time_us=attempt_time,
                      distance_to_swapper_km=d_ES, remote_loss=float(p_r_val))
    if p_EL == 0.0:  # No herald: its conditional state is undefined.
        parts = hp.empty_rates(num_level, mode, pool)
    else:
        parts = hp.overall_network_rates(
            get_ion_dm(*args), p_EL / attempt_time,
            t_merge_1=2.0 * (200.0 + 2.0 * t_signal),
            t_merge_2=2.0 * (200.0 + 4.0 * t_signal),
            num_level=num_level, t_meas=100.0,
            postprocessing=mode, pooling=pool, return_states=return_states)
    parts.update(distance_km=d, q=q_val, p_l=p_l_val, elementary=elementary)
    return parts


def get_key_rate(vars, d, p_l_val, num_level=2):
    """W-only at level 0; overall W + GHZ at levels 1 and 2."""
    return get_key_rate_components(vars, d, p_l_val, num_level)["total_bps"]


## Evaluate or optimize

`get_key_rate([0.01], d=100, p_l_val=0.1, num_level=2)` returns the total in bits/s. Use `get_key_rate_components(...)` for `W_bps`, `GHZ_bps`, and `total_bps`.

`optimize_rate(d=100, p_l_val=0.1, num_level=2)` returns the best evaluated parameters in `.x` and the negative total rate in `.fun`.


In [4]:
def optimize_rate(d, p_l_val, num_level=2, *, previous=None, **options):
    """Maximize the total, not the two components independently."""
    settings = {**SEARCH_OPTIONS, **options}
    return hp.optimize_q(get_key_rate, d, p_l_val, num_level,
                         q_bounds=bounds[0],
                         previous_q=None if previous is None else previous[0],
                         **settings)


def run_overall_sweeps(distances=None, losses=(0.01, 0.1, 0.5), levels=(0, 1, 2),
                       *, output_dir=None, search_options=None, verbose=True):
    def optimize(d, loss, level, previous):
        return optimize_rate(d, loss, level, previous=previous,
                             **(search_options or {}))
    return hp.run_rate_sweeps(
        get_key_rate_components, optimize, "atom_based",
        distances=distances, losses=losses, levels=levels,
        output_dir=output_dir, postprocessing=POSTPROCESSING,
        pooling=FINAL_POOLING, verbose=verbose)


## Optimize and save

Edit the grid below, then run this cell. One CSV per local loss is saved in `overall_rates_<postprocessing>_<pooling>/`.

`keyrate_bps_level_N` is already **W + GHZ**. The W/GHZ component columns use the same optimized parameters. Do not add a separate GHZ rate to the total again.

In [5]:
DISTANCES = np.linspace(0.0, 500.0, 20)
LOSSES = (0.01, 0.1, 0.5)
LEVELS = (0, 1, 2)

tables, optimizer_results, route_rates = run_overall_sweeps(
    distances=DISTANCES, losses=LOSSES, levels=LEVELS)

display(tables[0.1])


atom_based: loss=0.01, level=0, d=0 km, total=858.297 bps
atom_based: loss=0.01, level=0, d=26.3158 km, total=27.9777 bps
atom_based: loss=0.01, level=0, d=52.6316 km, total=5.5438 bps
atom_based: loss=0.01, level=0, d=78.9474 km, total=1.49824 bps
atom_based: loss=0.01, level=0, d=105.263 km, total=0.460683 bps
atom_based: loss=0.01, level=0, d=131.579 km, total=0.151321 bps
atom_based: loss=0.01, level=0, d=157.895 km, total=0.0516718 bps
atom_based: loss=0.01, level=0, d=184.211 km, total=0.0180984 bps
atom_based: loss=0.01, level=0, d=210.526 km, total=0.00645468 bps
atom_based: loss=0.01, level=0, d=236.842 km, total=0.00233365 bps
atom_based: loss=0.01, level=0, d=263.158 km, total=0.000852852 bps
atom_based: loss=0.01, level=0, d=289.474 km, total=0.000314425 bps
atom_based: loss=0.01, level=0, d=315.789 km, total=0.000116769 bps
atom_based: loss=0.01, level=0, d=342.105 km, total=4.36332e-05 bps
atom_based: loss=0.01, level=0, d=368.421 km, total=1.63912e-05 bps
atom_based: los

,distance_km,keyrate_bps_level_0,keyrate_W_bps_level_0,keyrate_GHZ_bps_level_0,opt_qs_level_0,p_accept_level_0,generation_time_us_level_0,round_time_us_level_0,keyrate_bps_level_1,keyrate_W_bps_level_1,...,p_accept_level_1,generation_time_us_level_1,round_time_us_level_1,keyrate_bps_level_2,keyrate_W_bps_level_2,keyrate_GHZ_bps_level_2,opt_qs_level_2,p_accept_level_2,generation_time_us_level_2,round_time_us_level_2
0,0.000000,3.652850e+02,3.652850e+02,0.0,0.158830,1.0,3.174589e+02,4.174589e+02,131.952187,64.443993,...,0.808482,1.617370e+03,1.717370e+03,24.728135,5.071931,19.656204,0.043795,0.665198,1.215850e+04,1.225850e+04
1,26.315789,2.431608e+01,2.431608e+01,0.0,0.055046,1.0,4.025890e+03,4.125890e+03,22.401486,10.511542,...,0.805039,7.553605e+03,7.653605e+03,6.958408,1.400409,5.557999,0.019199,0.664480,3.771367e+04,3.781367e+04
2,52.631579,4.967838e+00,4.967838e+00,0.0,0.043633,1.0,1.922866e+04,1.932866e+04,7.634488,3.547218,...,0.804277,2.093461e+04,2.103461e+04,3.147886,0.630118,2.517767,0.013371,0.664278,8.008469e+04,8.018469e+04
3,78.947368,1.354749e+00,1.354749e+00,0.0,0.040216,1.0,6.996490e+04,7.006490e+04,3.259785,1.508225,...,0.803962,4.782532e+04,4.792532e+04,1.700772,0.339573,1.361199,0.010783,0.664182,1.453708e+05,1.454708e+05
4,105.263158,4.179224e-01,4.179224e-01,0.0,0.038976,1.0,2.261442e+05,2.262442e+05,1.559631,0.720049,...,0.803804,9.867689e+04,9.877689e+04,1.012143,0.201783,0.810360,0.009347,0.664126,2.415196e+05,2.416196e+05
5,131.578947,1.374492e-01,1.374492e-01,0.0,0.038495,1.0,6.868220e+05,6.869220e+05,0.797902,0.367930,...,0.803716,1.914769e+05,1.915769e+05,0.640262,0.127522,0.512740,0.008452,0.664091,3.790179e+05,3.791179e+05
6,157.894737,4.695849e-02,4.695849e-02,0.0,0.038304,1.0,2.009434e+06,2.009534e+06,0.426139,0.196363,...,0.803664,3.569735e+05,3.570735e+05,0.422361,0.084068,0.338293,0.007853,0.664067,5.716875e+05,5.717875e+05
7,184.210526,1.645089e-02,1.645089e-02,0.0,0.038227,1.0,5.734801e+06,5.734901e+06,0.234351,0.107942,...,0.803634,6.474069e+05,6.475069e+05,0.287210,0.057141,0.230069,0.007434,0.664049,8.377086e+05,8.378086e+05
8,210.526316,5.867570e-03,5.867570e-03,0.0,0.038197,1.0,1.607745e+07,1.607755e+07,0.131595,0.060596,...,0.803615,1.151064e+06,1.151164e+06,0.199817,0.039741,0.160076,0.007131,0.664037,1.200945e+06,1.201045e+06
9,236.842105,2.121455e-03,2.121455e-03,0.0,0.038184,1.0,4.446606e+07,4.446616e+07,0.075042,0.034549,...,0.803603,2.016461e+06,2.016561e+06,0.141493,0.028134,0.113359,0.006907,0.664028,1.692663e+06,1.692763e+06
